# Deep Learning for Biological Modeling
### SIAM Life Sciences 2026 Mini-Tutorial — Deep Learning Section

1. **MNIST 3 vs. 8 — MLP**: A simple neural network baseline
2. **MNIST 3 vs. 8 — CNN**: Spatial inductive bias + interpretability via Grad-CAM
3. **PneumoniaMNIST — CNN**: Same pipeline on real biomedical chest X-rays

Run in **Google Colab**: execute the setup cell first, then proceed top to bottom.


## Setup

In [ ]:
!pip install medmnist -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import medmnist
from medmnist import INFO

print("TensorFlow version:", tf.__version__)


---
## Part 1 — MNIST (3 vs. 8): Classification with a Simple Neural Network

We restrict MNIST to two visually similar classes — **3** and **8** — as a binary classification task.


### 1.1 Data splits: 60 / 20 / 20

We partition all available samples into three non-overlapping sets:

$$\mathcal{D} = \mathcal{D}_{\text{train}} \cup \mathcal{D}_{\text{val}} \cup \mathcal{D}_{\text{test}}, \qquad |\text{splits}| \approx 60\%\ /\ 20\%\ /\ 20\%$$

- **Train** (60 %): weights are updated on this set via backpropagation.
- **Validation** (20 %): evaluated at each epoch to monitor generalisation — never used to update weights.
- **Test** (20 %): fully held out. **All reported accuracies come from this set only.**

> *Note:* the partner session uses an 80/20 train/test split for classical ML. We introduce a three-way split here because deep networks have many more hyperparameters to tune, making a dedicated validation set essential — and because it lets us use **early stopping** (see §1.3).


In [ ]:
(x_tr_full, y_tr_full), (x_te_full, y_te_full) = keras.datasets.mnist.load_data()

def filter_mnist(x, y, keep=(3, 8)):
    mask = (y == keep[0]) | (y == keep[1])
    return x[mask].astype("float32") / 255.0, (y[mask] == keep[1]).astype("float32")

x_all, y_all = map(np.concatenate, zip(
    filter_mnist(x_tr_full, y_tr_full),
    filter_mnist(x_te_full, y_te_full)
))

# Stratified 60 / 20 / 20
x_trainval, x_test,  y_trainval, y_test  = train_test_split(
    x_all, y_all, test_size=0.20, random_state=42, stratify=y_all)
x_train,    x_val,   y_train,    y_val   = train_test_split(
    x_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval)

print(f"Train:      {len(x_train):,}  |  Val: {len(x_val):,}  |  Test: {len(x_test):,} (held out)")


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
label_map = {0: "Digit 3", 1: "Digit 8"}
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray"); ax.set_title(label_map[int(y_train[i])], fontsize=9); ax.axis("off")
plt.suptitle("Sample training images", y=1.01); plt.tight_layout(); plt.show()


### 1.2 Overfitting and Early Stopping

A central challenge in deep learning is **overfitting**: the model memorises training data and loses the ability to generalise. Formally, we distinguish:

$$\mathcal{L}_{\text{train}}(\theta) = \frac{1}{|\mathcal{D}_{\text{train}}|} \sum_{i \in \mathcal{D}_{\text{train}}} \ell(f(x_i;\theta), y_i)$$
$$\mathcal{L}_{\text{val}}(\theta) = \frac{1}{|\mathcal{D}_{\text{val}}|} \sum_{i \in \mathcal{D}_{\text{val}}} \ell(f(x_i;\theta), y_i)$$

Overfitting occurs when $\mathcal{L}_{\text{train}}$ decreases while $\mathcal{L}_{\text{val}}$ stagnates or increases — the **generalisation gap** $\mathcal{L}_{\text{val}} - \mathcal{L}_{\text{train}}$ widens. Intuitively, a sufficiently expressive model can fit any finite training set, but this does not imply it has learned the underlying distribution.

**Early stopping** is the simplest regularisation strategy: halt training at epoch $t^*$ where:

$$t^* = \arg\min_{t}\ \mathcal{L}_{\text{val}}(\theta_t)$$

In practice we use a **patience** parameter $p$: stop if $\mathcal{L}_{\text{val}}$ has not improved for $p$ consecutive epochs, and restore the weights from the best epoch. This avoids both underfitting (stopping too early) and overfitting (stopping too late).


In [ ]:
# Early stopping callback — monitors validation loss
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,               # stop after 5 epochs with no improvement
    restore_best_weights=True,# revert to the epoch with lowest val_loss
    verbose=1
)
# We pass this to every model.fit() call below.


### 1.3 Multi-Layer Perceptron (MLP)

A single fully-connected layer maps $\mathbf{x} \in \mathbb{R}^n$ to $\mathbf{a} \in \mathbb{R}^m$ via:

$$\mathbf{a} = f\!\left(W\mathbf{x} + \mathbf{b}\right), \qquad W \in \mathbb{R}^{m \times n},\quad \mathbf{b} \in \mathbb{R}^m$$

Our two-layer network:
$$\mathbf{h} = \text{ReLU}(W_1 \mathbf{x} + \mathbf{b}_1), \quad W_1 \in \mathbb{R}^{64 \times 784}$$
$$\hat{y} = \sigma(\mathbf{w}_2^\top \mathbf{h} + b_2), \quad \mathbf{w}_2 \in \mathbb{R}^{64}$$

Training minimises the binary cross-entropy:

$$\mathcal{L}(\theta) = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\right]$$

via mini-batch SGD with the Adam optimiser (adaptive first-order method, $\eta = 0.001$ by default). We run up to 50 epochs, with early stopping on $\mathcal{L}_{\text{val}}$.


In [ ]:
def build_mlp():
    model = models.Sequential([
        layers.Flatten(input_shape=(28, 28)),
        layers.Dense(64, activation="relu"),
        layers.Dense(1,  activation="sigmoid"),
    ], name="MLP")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

mlp = build_mlp()
mlp.summary()


In [ ]:
mlp_history = mlp.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, ylabel in zip(axes, ["loss", "accuracy"], ["Loss", "Accuracy"]):
    ax.plot(mlp_history.history[metric],         label="Train")
    ax.plot(mlp_history.history[f"val_{metric}"], label="Validation", linestyle="--")
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.set_title(f"MLP — {ylabel}")
    ax.legend()
plt.suptitle("Early stopping halts training when val_loss stops improving", y=1.01)
plt.tight_layout(); plt.show()

mlp_loss, mlp_acc = mlp.evaluate(x_test, y_test, verbose=0)
print(f"MLP test accuracy (held-out 20%): {mlp_acc:.4f}")


---
## Part 2 — MNIST (3 vs. 8): CNN + Interpretability (Grad-CAM)

The MLP treats each image as a vector in $\mathbb{R}^{784}$, discarding spatial structure entirely. A **Convolutional Neural Network** instead encodes a *spatial inductive bias* by applying learned filters locally across the image.


### 2.1 Convolutional layers

A single filter $K \in \mathbb{R}^{k \times k}$ produces a feature map via discrete convolution:

$$(K * I)(i,j) = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} K(m,n)\cdot I(i+m,\,j+n)$$

With `padding="same"`, the output retains input dimensions. A layer with $C$ filters produces $C$ feature maps in parallel. **MaxPooling** retains the maximum activation in each $2 \times 2$ patch, reducing spatial resolution by $2\times$ while building translation invariance.

Our architecture:

| Layer | Output shape | Parameters |
|-------|-------------|------------|
| Conv2D (16 filters, 3×3, ReLU) | 28×28×16 | 160 |
| MaxPool 2×2 | 14×14×16 | 0 |
| Conv2D (32 filters, 3×3, ReLU) | 14×14×32 | 4,640 |
| MaxPool 2×2 | 7×7×32 | 0 |
| Flatten | 1,568 | 0 |
| Dense(64, ReLU) | 64 | 100,416 |
| Dense(1, sigmoid) | 1 | 65 |

The same `build_cnn()` function is reused for PneumoniaMNIST in Part 3 — only the learned parameters $\theta$ differ.


In [ ]:
def build_cnn(input_shape=(28, 28)):
    """Two-layer CNN — identical architecture used for MNIST and PneumoniaMNIST."""
    model = models.Sequential([
        layers.Reshape((28, 28, 1), input_shape=input_shape),
        layers.Conv2D(16, (3,3), activation="relu", padding="same", name="conv1"),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(32, (3,3), activation="relu", padding="same", name="conv2"),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ], name="CNN")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

cnn_mnist = build_cnn()
cnn_mnist.summary()


In [ ]:
cnn_mnist_history = cnn_mnist.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
cnn_loss, cnn_acc = cnn_mnist.evaluate(x_test, y_test, verbose=0)

print(f"MLP test accuracy (held-out 20%): {mlp_acc:.4f}")
print(f"CNN test accuracy (held-out 20%): {cnn_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, hist, title in zip(axes, [mlp_history, cnn_mnist_history], ["MLP", "CNN"]):
    ax.plot(hist.history["accuracy"],     label="Train")
    ax.plot(hist.history["val_accuracy"], label="Validation", linestyle="--")
    ax.set_title(f"{title} — accuracy"); ax.set_xlabel("Epoch"); ax.legend()
plt.tight_layout(); plt.show()


### 2.2 Interpretability: Grad-CAM

**Grad-CAM** (Selvaraju et al. 2017) produces a spatial heatmap of which image regions most influenced the prediction.

Let $A^k \in \mathbb{R}^{H \times W}$ be the $k$-th feature map of the last convolutional layer, and $y^c$ the score for class $c$. The importance weight of map $k$ is:

$$\alpha_k^c = \frac{1}{HW} \sum_{i=1}^{H}\sum_{j=1}^{W} \frac{\partial y^c}{\partial A^k_{ij}}$$

The heatmap is the ReLU of the weighted sum over all $K$ feature maps:

$$L^c_{\text{Grad-CAM}} = \text{ReLU}\!\left(\sum_{k=1}^{K} \alpha_k^c\, A^k\right) \in \mathbb{R}^{H \times W}$$

ReLU discards regions with negative influence (features that *suppress* class $c$). The $7 \times 7$ result is upsampled $4\times$ to the input resolution. A **smoothed** version applies a Gaussian kernel $G_\sigma$:

$$L^c_{\text{smooth}} = G_\sigma * L^c_{\text{Grad-CAM}}$$


In [ ]:
def grad_cam(model, image, last_conv_layer="conv2"):
    """
    Grad-CAM compatible with Keras 3 / TF 2.x.
    Runs a manual layer-by-layer forward pass and uses tape.watch()
    on the intermediate conv output — avoids relying on layer.output
    which requires the symbolic graph to have been built.
    """
    img_tensor = tf.cast(image[np.newaxis, ...], tf.float32)  # (1, 28, 28)

    # Identify target layer index once
    layer_names = [l.name for l in model.layers]
    target_idx  = layer_names.index(last_conv_layer)

    with tf.GradientTape() as tape:
        x = img_tensor
        conv_out = None
        for i, layer in enumerate(model.layers):
            x = layer(x)
            if i == target_idx:
                conv_out = x
                tape.watch(conv_out)   # explicitly track this tensor

        predictions = x               # final output after all layers
        loss = predictions[0, 0]      # binary sigmoid score

    grads = tape.gradient(loss, conv_out)            # (1, H, W, C)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))  # alpha_k

    heatmap = tf.reduce_sum(
        conv_out[0] * pooled_grads, axis=-1).numpy()  # weighted feature map sum
    heatmap = np.maximum(heatmap, 0)                  # ReLU
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    zoom = (image.shape[0] / heatmap.shape[0], image.shape[1] / heatmap.shape[1])
    return ndi.zoom(heatmap, zoom)


def smooth_cam(heatmap, sigma=2):
    """Gaussian-smooth: G_sigma * L."""
    return ndi.gaussian_filter(heatmap, sigma=sigma)


In [ ]:
def visualise_cam(model, images, labels, indices, label_map, title_prefix=""):
    fig, axes = plt.subplots(len(indices), 3, figsize=(9, 3.2 * len(indices)))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]
    for ax, ct in zip(axes[0], ["Original", "Grad-CAM", "Smoothed Grad-CAM"]):
        ax.set_title(ct, fontsize=10, fontweight="bold")
    for row, idx in enumerate(indices):
        img = images[idx]
        cam       = grad_cam(model, img)
        cam_smooth = smooth_cam(cam)
        axes[row, 0].imshow(img, cmap="gray")
        axes[row, 0].set_ylabel(label_map[int(labels[idx])], fontsize=9)
        for col, hm in enumerate([cam, cam_smooth], start=1):
            axes[row, col].imshow(img, cmap="gray")
            axes[row, col].imshow(hm, cmap="jet", alpha=0.45)
        for ax in axes[row]: ax.axis("off")
    plt.suptitle(f"{title_prefix}Grad-CAM", y=1.01)
    plt.tight_layout(); plt.show()

idx_3 = np.where(y_test == 0)[0][5]
idx_8 = np.where(y_test == 1)[0][5]
label_map_mnist = {0: "Digit 3", 1: "Digit 8"}

visualise_cam(cnn_mnist, x_test, y_test,
              indices=[idx_3, idx_8],
              label_map=label_map_mnist,
              title_prefix="MNIST CNN — ")


**What to look for:**
- The raw Grad-CAM reflects the $7 \times 7$ feature map resolution of `conv2`, upsampled $4\times$.
- Smoothing ($\sigma = 2$) suppresses high-frequency artefacts from the upsampling.
- For **3**: does the network attend to the two open arcs? For **8**: to the two closed loops?
- If you ran Grad-CAM on the MLP, its maps would scatter across disconnected pixels — a direct consequence of discarding the image's 2-D metric structure via flattening.


---
## Part 3 — PneumoniaMNIST: Interpretability in Medical Imaging

We apply the **identical architecture and Grad-CAM pipeline** to a real biomedical dataset.

**PneumoniaMNIST** (MedMNIST v2; Yang et al. 2023) contains pediatric chest X-rays downsampled to $28 \times 28$, labelled Normal ($y=0$) or Pneumonia ($y=1$). The dataset provides its own train/val/test splits.

The central question is no longer only accuracy, but: **does the network attend to clinically relevant anatomy?**


### 3.1 Load and preview

In [ ]:
DataClass = getattr(medmnist, INFO['pneumoniamnist']['python_class'])

splits = {}
for split in ('train', 'val', 'test'):
    ds = DataClass(split=split, download=True)
    imgs   = np.array([item[0] for item in ds], dtype="float32") / 255.0
    labels = np.array([item[1][0] for item in ds], dtype="float32")
    splits[split] = (imgs.squeeze(), labels)

x_pneu_train, y_pneu_train = splits['train']
x_pneu_val,   y_pneu_val   = splits['val']
x_pneu_test,  y_pneu_test  = splits['test']

total = sum(len(v[0]) for v in splits.values())
for name, (x, y) in splits.items():
    print(f"{name:6s}: {len(x):,} ({100*len(x)/total:.0f}%)  "
          f"Normal: {(y==0).sum()}  Pneumonia: {(y==1).sum()}")


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
label_map_pneu = {0: "Normal", 1: "Pneumonia"}
for row, (cls, title) in enumerate([(0, "Normal"), (1, "Pneumonia")]):
    for col, idx in enumerate(np.where(y_pneu_train == cls)[0][:5]):
        ax = axes[row, col]
        ax.imshow(x_pneu_train[idx], cmap="gray")
        if col == 0: ax.set_ylabel(title, fontsize=10)
        ax.axis("off")
plt.suptitle("PneumoniaMNIST samples"); plt.tight_layout(); plt.show()


### 3.2 Train the CNN (same architecture)

We call `build_cnn()` without modification. What changes between Part 2 and Part 3 is entirely contained in the learned weight matrices $\theta = \{W_\ell, \mathbf{b}_\ell\}_\ell$ — the functional form $f(\cdot;\theta)$ is identical.

> **Class imbalance note:** pneumonia cases outnumber normals roughly 3:1. For a tutorial we accept this without correction; in practice one would use class-weighted loss or oversampling. The imbalance means raw accuracy can be misleading — a model predicting "always pneumonia" would score ~75%. Keep this in mind when reading the results.


In [ ]:
cnn_pneu = build_cnn()

cnn_pneu_history = cnn_pneu.fit(
    x_pneu_train, y_pneu_train,
    validation_data=(x_pneu_val, y_pneu_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

pneu_loss, pneu_acc = cnn_pneu.evaluate(x_pneu_test, y_pneu_test, verbose=0)
print(f"\nPneumoniaMNIST CNN test accuracy (held-out set): {pneu_acc:.4f}")


### 3.3 Smoothed Grad-CAM on chest X-rays

Recalling the formula from Part 2:

$$L^c_{\text{Grad-CAM}} = \text{ReLU}\!\left(\sum_k \alpha_k^c A^k\right), \qquad \alpha_k^c = \frac{1}{HW}\sum_{i,j}\frac{\partial y^c}{\partial A^k_{ij}}$$

At $28 \times 28$ input, `conv2` outputs $7 \times 7$ feature maps ($H = W = 7$, $K = 32$). The heatmap is therefore coarse; smoothing with $G_\sigma$ partially compensates.

**Clinical question:** for a Pneumonia prediction, does $L^c_{\text{Grad-CAM}}$ localise to the lung fields — or does it activate over the cardiac silhouette, ribs, or diaphragm? The latter would suggest the model has learned a spurious shortcut rather than pneumonia-specific pathology.


In [ ]:
idx_norm_t = np.where(y_pneu_test == 0)[0][2]
idx_pneu_t = np.where(y_pneu_test == 1)[0][2]

visualise_cam(cnn_pneu, x_pneu_test, y_pneu_test,
              indices=[idx_norm_t, idx_pneu_t],
              label_map=label_map_pneu,
              title_prefix="PneumoniaMNIST CNN — ")


**Discussion questions:**
1. For a pneumonia X-ray, does the heatmap concentrate over the lung fields? If it strongly activates over the heart shadow or costophrenic angles, what does that imply about the model's reasoning?
2. The MLP saliency maps are spatially incoherent — explain this in terms of the flattening operation's destruction of the image's 2-D metric structure.
3. At $28 \times 28$, the dataset discards considerable anatomical detail. How might the Grad-CAM maps differ on full-resolution X-rays ($1024 \times 1024$)?
4. Class imbalance affects not only accuracy but also the gradient landscape during training. How might you modify $\mathcal{L}(\theta)$ to account for this? (Hint: weighted cross-entropy, $\mathcal{L} = -\sum_i w_{y_i} \ell_i$.)


---
## Summary

| Model | Dataset | Test accuracy |
|-------|---------|---------------|
| MLP (1 hidden layer, 50k params) | MNIST 3 vs 8 | *(printed above)* |
| CNN (2 conv layers, 105k params) | MNIST 3 vs 8 | *(printed above)* |
| CNN (same architecture)          | PneumoniaMNIST | *(printed above)* |

**Key takeaways:**
- CNNs encode a *spatial inductive bias*: filter weights are shared across all positions (translation equivariance). This is why they generalise better on image data than MLPs of comparable parameter count.
- Early stopping uses the validation loss as a proxy for the generalisation gap; it stops training at $t^* = \arg\min_t \mathcal{L}_{\text{val}}(\theta_t)$ and restores the best weights.
- Grad-CAM turns $\nabla_{A^k} y^c$ into an interpretability tool at no additional training cost — the same gradient computation used in backpropagation.
- Interpretability is most valuable in biomedical settings because accuracy alone is insufficient: a clinician needs to audit *how* the model reaches its decision.

**Next:** Deep-Symbolic Optimization (DSO) for learning ODEs from data.
